In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/sharankumare/mcq-train/train.csv
/kaggle/input/datasets/sharankumare/mcq-test/test.csv


In [2]:
train = pd.read_csv("/kaggle/input/datasets/sharankumare/mcq-train/train.csv")

In [3]:
print(train['answer'].value_counts(ascending=False))
freq_dist = list(train['answer'].value_counts(ascending=False))
print(freq_dist[0]+freq_dist[-1])

answer
B    490
C    459
A    369
D    358
E    324
Name: count, dtype: int64
814


In [4]:
import string

vocab = set()

for text in train["prompt"]:
    # lowercase
    text = str(text).lower()

    # remove standard punctuation
    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )

    # split by whitespace
    words = text.split()

    vocab.update(words)

print("Vocabulary Size:", len(vocab))

Vocabulary Size: 859


In [5]:
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS
# Row ID 1
text = train.loc[train["id"] == 1, "prompt"].iloc[0]

# lowercase + remove punctuation
text = text.lower()
text = text.translate(str.maketrans('', '', string.punctuation))

# tokenize
words = text.split()

# remove stopwords
filtered_words = [w for w in words if w not in ENGLISH_STOP_WORDS]

print(filtered_words)
print("Count:", len(filtered_words))

['pick', 'best', 'possible', 'answer', 'martin', 'heideggers', 'view', 'relationship', 'time', 'human', 'existence', 'listed', 'options']
Count: 13


In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer
combined_text = (
    train["prompt"].fillna("") + " " +
    train["A"].fillna("") + " " +
    train["B"].fillna("") + " " +
    train["C"].fillna("") + " " +
    train["D"].fillna("") + " " +
    train["E"].fillna("")
)

vectorizer = TfidfVectorizer(stop_words="english")

X = vectorizer.fit_transform(combined_text)

print("Vocabulary Size:", len(vectorizer.get_feature_names_out()))

Vocabulary Size: 2762


In [7]:
from sklearn.metrics.pairwise import cosine_similarity

row = train.iloc[0]   # Row ID 1 if first row

prompt_vec = vectorizer.transform([row["prompt"]])
A_vec = vectorizer.transform([row["A"]])

score = cosine_similarity(prompt_vec, A_vec)[0][0]

print(round(score, 4))

0.272


In [8]:
correct = 0

for _, row in train.iterrows():

    prompt_vec = vectorizer.transform([row["prompt"]])

    scores = {}

    for option in ["A", "B", "C", "D", "E"]:

        option_vec = vectorizer.transform([row[option]])

        scores[option] = cosine_similarity(
            prompt_vec,
            option_vec
        )[0][0]

    predicted = max(scores, key=scores.get) # highest cosine similarity

    if predicted == row["answer"]:
        correct += 1

accuracy = (correct / len(train)) * 100

print(f"Accuracy: {accuracy:.2f}%")

Accuracy: 13.55%


In [9]:
def map_at_3(actual, predicted):
    try:
        rank = predicted.index(actual) + 1
        return 1.0 / rank
    except ValueError:
        return 0.0

actual = "C"
predicted = ["C", "A", "B"]

score = map_at_3(actual, predicted)

print(score)

1.0


In [10]:
actual = "B"
predicted = ["D", "B", "E"]

score = map_at_3(actual, predicted)

print(score)

0.5


In [11]:
top3 = train["answer"].value_counts().index[:3].tolist()

prediction = top3

score = 0

for ans in train["answer"]:

    if ans == prediction[0]:
        score += 1.0

    elif ans == prediction[1]:
        score += 0.5

    elif ans == prediction[2]:
        score += 1/3

map3 = score / len(train)

print("Top 3:", prediction)
print("MAP@3:", round(map3, 4))

Top 3: ['B', 'C', 'A']
MAP@3: 0.4213
